# Автоматическое создание detection rules

**Evidence → DetectionSpec → ALERT candidate → validation → replay.**

A: известный синтетический IOC. B: неизвестный статистический DNS-паттерн.
IF ищет необычное; Qwen предлагает гипотезу; Python ограничивает правило; Suricata исполняет;
только evaluator читает ground truth. Кандидаты автоматически не активируются.

Нужны работающие Docker Desktop и локальный Ollama с уже установленной Qwen 0.6B.
Короткий профиль: 1% suspicious events, окна 5 секунд; realistic: 0.1%, окна 60 секунд.
До занятия выполните `make prepare`: capture отделён от показа.
Каждый Run All вызывает LLM и Suricata заново на подготовленном трафике.
`RULE_DEMO_RUN` выбирает источник capture; новые ответы сохраняются в новом каталоге.


In [1]:
import json
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

from ids_ml_lab.features import DNS_WINDOW_FEATURES
from ids_ml_lab.ollama import OllamaAdapter
from ids_ml_lab.rule_generation import (
    DetectionSpec,
    copy_capture,
    discover,
    ioc_evidence,
    read_jsonl,
    review_behavioral_spec,
    write_candidate,
)
from ids_ml_lab.rule_validation import evaluate_run

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Выберите Python проектной .venv"
os.chdir(ROOT)


DEFAULT_RUN = Path("artifacts/rule_generation/prepared")
if not DEFAULT_RUN.exists():
    DEFAULT_RUN = Path("artifacts/rule_generation/qwen06-complete")
SOURCE = Path(os.getenv("RULE_DEMO_RUN", str(DEFAULT_RUN)))
OFFLINE = os.getenv("LECTURE_OFFLINE") == "1"
RUN_BEHAVIORAL = os.getenv("LECTURE_BEHAVIORAL") == "1"
assert (SOURCE / "settings.json").exists(), "Сначала выполните make prepare; см. README"
settings = json.loads((SOURCE / "settings.json").read_text())
RUN = SOURCE if OFFLINE else Path("artifacts/rule_generation/demo") / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
if not OFFLINE:
    copy_capture(SOURCE, RUN, ("A", "B") if RUN_BEHAVIORAL else ("A",))
adapter = OllamaAdapter(settings)
if not OFFLINE:
    print(json.dumps(adapter.health(), indent=2))
print("MODE:", "OFFLINE: saved responses/replay" if OFFLINE else "LIVE LLM + Suricata replay")
print("CAPTURE:", SOURCE, "OUTPUT:", RUN)


{
  "model": "hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0",
  "digest": "54d053a1b435c1f0cbe74f83da1a2365857b61e8ca04751d37c491ec15f32ab7",
  "ollama_version": "0.35.1",
  "local_only": true
}
MODE: LIVE LLM + Suricata replay
CAPTURE: artifacts/rule_generation/prepared OUTPUT: artifacts/rule_generation/demo/20261006T163113390589Z


## 1. Реальный traffic от существующего generator → Zeek + PCAP

In [2]:
a_raw = read_jsonl(RUN / "A/raw/dns.log")
baseline_raw = read_jsonl(RUN / "baseline/raw/dns.log")
if RUN_BEHAVIORAL:
    b_raw = read_jsonl(RUN / "B/raw/dns.log")
display(pd.DataFrame(a_raw)[["ts", "id.orig_h", "id.orig_p", "query", "rcode_name"]].head())
print("Captured DNS events:", {"baseline": len(baseline_raw), "A": len(a_raw)})


,ts,id.orig_h,id.orig_p,query,rcode_name
0,1.791304e+09,172.18.0.3,20002,evt-b45ed1f03139d32c93cd59bf5c941cf0.db.backen...,NOERROR
1,1.791304e+09,172.18.0.3,20002,evt-a2bc372f7412b29347294739614ff3d7.db.backen...,NOERROR
2,1.791304e+09,172.18.0.3,20001,evt-2bcfbe01a28defe39bf0027312476f57.api.dev.test,NOERROR
3,1.791304e+09,172.18.0.3,20001,evt-5304317faf42e12f3838b3268e944239.api.dev.test,NOERROR
4,1.791304e+09,172.18.0.3,20001,evt-f16287e4e9c349e03602f8ac10f1bc81.api.dev.test,NOERROR


Captured DNS events: {'baseline': 5409, 'A': 5411}


## 2. Scenario A: relevant evidence без ground truth

In [3]:
a_evidence = ioc_evidence(a_raw, baseline_raw)
display(pd.DataFrame(a_evidence["observed_names"]))
print("Выбор evidence: имена, отсутствующие в baseline, без меток; новизна не доказывает атаку.")

,query,count
0,telemetry-update.bad-example.test,60


Выбор evidence: имена, отсутствующие в baseline, без меток; новизна не доказывает атаку.


## 3. Локальная Qwen → schema-validated DetectionSpec

Проверяем не только JSON. Модель не имеет threat intelligence: заявления о known malicious domains
не подтверждены. Текст hypothesis и test_plan требует проверки человеком; автоматическая проверка
правила опирается на условия, настоящий syntax check и replay.

In [4]:
def get_spec(evidence, directory):
    path = directory / "llm/detection_spec.json"
    if OFFLINE:
        print("Используем сохранённый валидированный spec:", path)
        return DetectionSpec.model_validate_json(path.read_text())
    return adapter.generate(evidence, Path("prompts/detection_rule_system.txt"), directory / "llm")

a_spec = get_spec(a_evidence, RUN / "A")
print(a_spec.model_dump_json(indent=2))
print("SURICATA COMPATIBLE:", a_spec.suricata_compatible)

{
  "name": "telemetry-update.bad-example.test",
  "description": "A DNS query indicating a potential threat, with a count of 60.",
  "evidence": [
    "The observed domain name 'telemetry-update.bad-example.test' is a narrow and stable indicator. It does not match any known malicious domains, making it a strong candidate for detection.",
    "This evidence suggests that the domain may be associated with automated or benign activity, though further investigation is needed to confirm its actual impact."
  ],
  "hypothesis": "The observed activity could be related to automated requests or benign automation, but no definitive proof of an attack has been established.",
  "rule_type": "ioc",
  "protocol": "dns",
  "source_scope": "any",
  "destination_scope": "any",
  "conditions": [
    {
      "field": "dns.query",
      "operator": "endswith",
      "value": ".telemetry-update.bad-example.test"
    }
  ],
  "severity": "low",
  "suricata_compatible": true,
  "requires_correlation": false

## 4. Детерминированный renderer → candidate ALERT rule

In [5]:
a_rule_path = RUN / "A/rules/candidate.rules"
if not a_rule_path.exists():
    a_rule_path = write_candidate(a_spec, RUN / "A/rules")
print(a_rule_path.read_text())

alert dns any any -> any any (msg:"LOCAL GENERATED candidate"; dns.query; content:".telemetry-update.bad-example.test"; nocase; endswith; sid:9900001; rev:1;)



## 5. Suricata syntax validation → настоящий PCAP replay → quality gate

In [6]:
a_report_path = RUN / "A/quality_report.json"
a_report = json.loads(a_report_path.read_text()) if a_report_path.exists() else evaluate_run(RUN / "A", settings)
print("SYNTAX:", "PASS" if a_report["syntax_valid"] else "FAIL")
print(f"Attack detected: {a_report['detected_attack_events']}/{a_report['expected_attack_events']}")
print("Benign matches:", a_report["benign_matches"], "Alerts:", a_report["alerts_total"])
print("DECISION:", a_report["decision"], "— candidate only; human approval required")
display(pd.Series(a_report).to_frame("value"))
# Этот файл записала Suricata при replay именно candidate.rules.
eve = read_jsonl(RUN / "A/replay/eve.json")
alerts = [event for event in eve if event.get("event_type") == "alert"]
display(pd.DataFrame([{"timestamp": event["timestamp"],
                       "sid": event["alert"]["signature_id"],
                       "message": event["alert"]["signature"]} for event in alerts[:5]]))
print("Suricata EVE:", RUN / "A/replay/eve.json")


SYNTAX: PASS
Attack detected: 60/60
Benign matches: 0 Alerts: 60
DECISION: ACCEPT_CANDIDATE — candidate only; human approval required


,value
generated_rule,"alert dns any any -> any any (msg:""LOCAL GENER..."
syntax_valid,True
expected_attack_events,60
detected_attack_events,60
missed_attack_events,0
benign_events,5940
benign_matches,0
alerts_total,60
matched_events,60
alerts_per_1000_events,10.0


,timestamp,sid,message
0,2026-10-06T16:29:27.847061+0000,9900001,LOCAL GENERATED candidate
1,2026-10-06T16:29:28.655776+0000,9900001,LOCAL GENERATED candidate
2,2026-10-06T16:29:28.722223+0000,9900001,LOCAL GENERATED candidate
3,2026-10-06T16:29:28.866122+0000,9900001,LOCAL GENERATED candidate
4,2026-10-06T16:29:28.952173+0000,9900001,LOCAL GENERATED candidate


Suricata EVE: artifacts/rule_generation/demo/20261006T163113390589Z/A/replay/eve.json


## 6. Валидный синтаксис не гарантирует качество
Контрольное слишком широкое правило `.test` написано локально для сравнения; оно не является ответом Qwen.

In [7]:
broad = json.loads((RUN / "A/broad_quality_report.json").read_text())
display(pd.DataFrame([a_report, broad], index=["Qwen candidate", "Broad negative control"])[
    ["syntax_valid", "detected_attack_events", "benign_matches", "precision", "decision"]])

,syntax_valid,detected_attack_events,benign_matches,precision,decision
Qwen candidate,True,60,0,1.000000,ACCEPT_CANDIDATE
Broad negative control,True,60,5351,0.011089,REJECT



## 7. Scenario B: 60-секундные окна (5 секунд в ускоренном demo)
Client entity — IP и стабильный UDP source port сессии. Префикс `evt-…` удаляется из DNS-статистик. Labels не читаются.

In [8]:
if RUN_BEHAVIORAL:
    b_summary = discover(baseline_raw, b_raw, settings, RUN / "B")
    base_features = pd.read_parquet(RUN / "B/baseline_features.parquet")
    b_features = pd.read_parquet(RUN / "B/features.parquet")
    scores = pd.read_parquet(RUN / "B/isolation_forest_scores.parquet")
    inspect_window = scores.head(settings["top_anomalies"]).sort_values("unique_subdomains", ascending=False).iloc[0]
    print("Показываем окно из top-N с наибольшим числом unique subdomains; rank:", inspect_window["rank"])
    display(pd.DataFrame({"baseline median": base_features[DNS_WINDOW_FEATURES].median(),
                          "selected top-N window": inspect_window[DNS_WINDOW_FEATURES]}))
else:
    print("Optional behavioral section: set LECTURE_BEHAVIORAL=1 to run")


Optional behavioral section: set LECTURE_BEHAVIORAL=1 to run



## 8. Isolation Forest: top-N ranking
Fit использует только отдельный baseline capture. Ни labels, ни подбор порога по ground truth не нужны; score не является вероятностью атаки.

In [9]:
if RUN_BEHAVIORAL:
    display(scores.head(settings["top_anomalies"]))
    print(json.dumps(b_summary, indent=2))
else:
    print("Optional behavioral section: set LECTURE_BEHAVIORAL=1 to run")


Optional behavioral section: set LECTURE_BEHAVIORAL=1 to run


**Дополнительный раздел — вне обязательного показа.**

## 9. Qwen интерпретирует окна и формализует behavioral hypothesis

In [10]:
if RUN_BEHAVIORAL:
    b_spec = get_spec(b_summary, RUN / "B")
    print(b_spec.model_dump_json(indent=2))
    assert b_spec.rule_type == "behavioral" and b_spec.requires_correlation and not b_spec.suricata_compatible
    print("SURICATA COMPATIBLE: NO")
    print("REASON: Requires window aggregation of unique names, entropy, NXDOMAIN ratio or bursts.")
    print("RECOMMENDATION: Correlation detector; manual review.")
    print("HYPOTHESIS:", b_spec.hypothesis)
    review = review_behavioral_spec(b_spec, base_features, scores.head(settings["top_anomalies"]))
    print(json.dumps(review, indent=2))
    report = {"reason": "Window aggregation needs a correlation detector", "evidence_review": review, "decision": "REQUIRES_REVIEW", "status": "manual_review", "suricata_compatible": False,
              "requires_correlation": True, "specification": b_spec.model_dump()}
    (RUN / "B/quality_report.json").write_text(json.dumps(report, indent=2))
else:
    print("Optional behavioral section: set LECTURE_BEHAVIORAL=1 to run")
adapter.close()


Optional behavioral section: set LECTURE_BEHAVIORAL=1 to run


## Результат занятия

**AI proposes · code constrains · engine validates · data tests · human approves.**

A показывает создание и проверку кандидата. B показывает честную границу нативной сигнатуры.
Correlation engine и deployment не входят в практику. Артефакты содержат исходный PCAP,
телеметрию, отдельный ground truth, features/ranking, exact model/digest, prompt, ответы и отчёты.
Редкость событий, скорости и временные окна задаются в `config/rule_generation.yaml`.
Seed воспроизводит запросы и расписание; реальные timestamps/PCAP и ответы LLM могут отличаться.